# 24h-ahead forecasting baselines

This notebook compares two approaches for forecasting `preis` 24 hours ahead:

- Linear Regression with lag and calendar features
- SARIMAX on the hourly `preis` series

Both models are evaluated with the same metrics so the results are directly comparable.

In [ ]:
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

In [ ]:
DATA_PATH = "hist_data.csv"
FORECAST_HORIZON = 24  # 24 hours ahead

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(df.shape)
df.head()

In [ ]:
model_df = df[["timestamp", "preis"]].copy()

# Target: the price 24 hours after the current timestamp.
model_df["target_preis_24h"] = model_df["preis"].shift(-FORECAST_HORIZON)

# Calendar features from the current timestamp.
model_df["hour"] = model_df["timestamp"].dt.hour
model_df["day_of_week"] = model_df["timestamp"].dt.dayofweek
model_df["month"] = model_df["timestamp"].dt.month
model_df["is_weekend"] = (model_df["day_of_week"] >= 5).astype(int)

# Lag features from historical price only.
lag_hours = [1, 2, 3, 6, 12, 24, 48, 72, 168]
for lag in lag_hours:
    model_df[f"preis_lag_{lag}"] = model_df["preis"].shift(lag)

# Rolling features based only on already-observed values.
model_df["preis_roll_mean_24"] = model_df["preis"].shift(1).rolling(24).mean()
model_df["preis_roll_std_24"] = model_df["preis"].shift(1).rolling(24).std()
model_df["preis_roll_mean_168"] = model_df["preis"].shift(1).rolling(168).mean()

model_df = model_df.dropna().reset_index(drop=True)
model_df.head()

In [ ]:
feature_cols = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    *[f"preis_lag_{lag}" for lag in lag_hours],
    "preis_roll_mean_24",
    "preis_roll_std_24",
    "preis_roll_mean_168",
]

X = model_df[feature_cols]
y = model_df["target_preis_24h"]

# Chronological split for time series.
split_idx = int(len(model_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
ts_test = model_df["timestamp"].iloc[split_idx:].reset_index(drop=True)

print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred_test = linear_model.predict(X_test)

linear_metrics = {
    "model": "LinearRegression",
    "MAE": mean_absolute_error(y_test, linear_pred_test),
    "RMSE": np.sqrt(mean_squared_error(y_test, linear_pred_test)),
    "R2": r2_score(y_test, linear_pred_test),
}

pd.DataFrame([linear_metrics]).round(3)

In [ ]:
linear_results = pd.DataFrame(
    {
        "timestamp": ts_test,
        "actual_preis_24h": y_test.reset_index(drop=True),
        "linear_pred_preis_24h": linear_pred_test,
    }
)

linear_results["linear_abs_error"] = (
    linear_results["actual_preis_24h"] - linear_results["linear_pred_preis_24h"]
).abs()

linear_results.head(20)

## SARIMAX baseline

For SARIMAX we forecast the raw hourly `preis` series 24 steps ahead. The rolling evaluation below updates the model with each newly observed hour in the test window and records the 24-hour-ahead forecast for the same timestamps used above.

In [ ]:
sarimax_order = (1, 0, 1)
sarimax_seasonal_order = (1, 0, 1, 24)

price_series = df.set_index("timestamp")["preis"].asfreq("h")
test_timestamps = ts_test.tolist()

# Start from the first evaluation timestamp and forecast 24h ahead in a rolling way.
first_test_idx = price_series.index.get_loc(test_timestamps[0])
initial_history = price_series.iloc[: first_test_idx + 1]

sarimax_model = SARIMAX(
    initial_history,
    order=sarimax_order,
    seasonal_order=sarimax_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_results = sarimax_model.fit(disp=False)

sarimax_preds = []
current_history_end = first_test_idx

for ts in test_timestamps:
    ts_idx = price_series.index.get_loc(ts)

    while current_history_end < ts_idx:
        current_history_end += 1
        sarimax_results = sarimax_results.append(
            price_series.iloc[current_history_end : current_history_end + 1],
            refit=False,
        )

    forecast = sarimax_results.forecast(steps=FORECAST_HORIZON)
    sarimax_preds.append(float(forecast.iloc[-1]))

sarimax_metrics = {
    "model": "SARIMAX",
    "MAE": mean_absolute_error(y_test, sarimax_preds),
    "RMSE": np.sqrt(mean_squared_error(y_test, sarimax_preds)),
    "R2": r2_score(y_test, sarimax_preds),
}

pd.DataFrame([sarimax_metrics]).round(3)

In [ ]:
comparison = linear_results.copy()
comparison["sarimax_pred_preis_24h"] = sarimax_preds
comparison["sarimax_abs_error"] = (
    comparison["actual_preis_24h"] - comparison["sarimax_pred_preis_24h"]
).abs()

comparison.head(20)

In [ ]:
metrics_comparison = pd.DataFrame([linear_metrics, sarimax_metrics]).round(3)
metrics_comparison

In [ ]:
future_linear_row = model_df.iloc[[-1]].copy()
linear_future_pred = linear_model.predict(future_linear_row[feature_cols])[0]

future_sarimax_forecast = sarimax_results.forecast(steps=FORECAST_HORIZON)
sarimax_future_pred = float(future_sarimax_forecast.iloc[-1])

forecast_timestamp = future_linear_row["timestamp"].iloc[0] + pd.Timedelta(hours=FORECAST_HORIZON)

pd.DataFrame(
    [
        {"model": "LinearRegression", "forecast_timestamp": forecast_timestamp, "predicted_preis_24h": linear_future_pred},
        {"model": "SARIMAX", "forecast_timestamp": forecast_timestamp, "predicted_preis_24h": sarimax_future_pred},
    ]
).round(3)